In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)  # use Runtime -> Change runtime type -> T4 GPU for fast training

In [ ]:
# Load IMDB reviews (small subsets so training finishes in a few minutes)
ds = load_dataset("imdb")
train_ds = ds["train"].shuffle(seed=42).select(range(2000))
test_ds = ds["test"].shuffle(seed=42).select(range(500))
print(train_ds)
print("Sample:", train_ds[0]["text"][:200], "| label:", train_ds[0]["label"])

In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_tok = train_ds.map(tokenize, batched=True)
test_tok = test_ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label={0: "NEGATIVE", 1: "POSITIVE"}, label2id={"NEGATIVE": 0, "POSITIVE": 1})

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds), "f1": f1_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir="bert_imdb",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=test_tok,
                  data_collator=collator, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# Evaluation
out = trainer.predict(test_tok)
y_pred = np.argmax(out.predictions, axis=1)
print("Test metrics:", out.metrics)
print(classification_report(out.label_ids, y_pred, target_names=["NEGATIVE", "POSITIVE"]))
ConfusionMatrixDisplay.from_predictions(out.label_ids, y_pred, display_labels=["NEG", "POS"], cmap="Blues")
plt.title("Fine-tuned BERT - confusion matrix"); plt.show()

In [ ]:
# Training loss curve
logs = [l for l in trainer.state.log_history if "loss" in l]
plt.plot([l["step"] for l in logs], [l["loss"] for l in logs], marker="o")
plt.xlabel("step"); plt.ylabel("training loss"); plt.title("Fine-tuning loss"); plt.show()

In [ ]:
# Predict on new text
def predict(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
    return probs

samples = ["An absolutely brilliant film with a touching story.",
           "Boring plot, terrible acting, I wanted to leave the theatre.",
           "It was okay, nothing special but not bad either."]
for s, p in zip(samples, predict(samples)):
    print(f"{model.config.id2label[int(p.argmax())]:8s} ({p.max():.2f}) <- {s}")

trainer.save_model("bert_imdb_final")
tokenizer.save_pretrained("bert_imdb_final")